<a href="https://colab.research.google.com/github/shaipshiverya/Data_Analysis_python_projects/blob/main/cafe_cleaning_data_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Introduction

This project analyzes dirty_cafe_sales.csv, a real-world dataset capturing transactional records from a cafe — likely containing inconsistencies such as missing values, incorrect formats, or duplicate entries. Using Google Colab, we’ll clean, explore, and visualize the data with Python libraries like pandas, matplotlib, and seaborn. The goal is to uncover actionable insights — such as top-selling items, peak sales hours, or customer spending trends — to support better business decisions.



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
df = pd.read_csv('/content/dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [ ]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


# Key Findings

 After reviewing the dataset information, we identified two main issues:

 Many columns contain null values.

 Some columns that should be numeric are stored as object data type.

 We will address these problems now.

In [ ]:
df.nunique()

,0
Transaction ID,10000
Item,10
Quantity,7
Price Per Unit,8
Total Spent,19
Payment Method,5
Location,4
Transaction Date,367


# Convert Quantity to Integer type and fill values

In [ ]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors = 'coerce')

# Convert Price per unit to Integer type

In [ ]:
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'] , errors = 'coerce')

# Convert Total spent to Integer type

In [ ]:
df['Total Spent'] = pd.to_numeric(df['Total Spent'] ,  errors = 'coerce')

# Convert Transaction Date to Date type

In [ ]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors = 'coerce')

# Duplicate Data Check

We checked the dataset for duplicate rows and found that there are no duplicates

In [ ]:
df.duplicated().sum()

np.int64(0)

# Filling Missing Values

The dataset contains many null values.
In this step, we will handle them by filling or imputing the missing data.

In [ ]:
df.isna().sum().sum()

np.int64(8151)

# Lets tackle ITEM COLUMN  first

In [ ]:
df[df['Item'].isna()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
30,TXN_1736287,NaN,5.0,2.0,10.0,Digital Wallet,NaN,2023-06-02
61,TXN_8051289,NaN,1.0,3.0,3.0,NaN,In-store,2023-10-09
72,TXN_6044979,NaN,1.0,1.0,1.0,Cash,In-store,2023-12-08
89,TXN_4132730,NaN,5.0,1.0,5.0,NaN,In-store,2023-03-12
...,...,...,...,...,...,...,...,...
9820,TXN_8751702,NaN,5.0,NaN,15.0,Cash,NaN,2023-02-13
9855,TXN_3740505,NaN,2.0,1.5,3.0,NaN,NaN,2023-11-21
9876,TXN_3105633,NaN,1.0,2.0,2.0,NaN,In-store,2023-03-30
9885,TXN_4659954,NaN,3.0,4.0,12.0,Credit Card,In-store,NaT


In [ ]:
x = df['Item'].unique()
pd.DataFrame(x)

,0
0,Coffee
1,Cake
2,Cookie
3,Salad
4,Smoothie
5,UNKNOWN
6,Sandwich
7,NaN
8,ERROR
9,Juice


# Handling Nulls in 'Item'

There are 8 unique values in the 'Item' column, with some missing (NaN) values.
We will use basic math and insights from the data to fill these missing entries.

In [ ]:
z = df[['Item','Price Per Unit']].value_counts().head(8).reset_index()
z

,Item,Price Per Unit,count
0,Juice,3.0,1110
1,Coffee,2.0,1108
2,Cake,3.0,1085
3,Salad,5.0,1082
4,Sandwich,4.0,1082
5,Smoothie,4.0,1036
6,Cookie,1.0,1026
7,Tea,1.5,1023


# Fill Missing Prices Using Item Names

The dataset has 8 unique items, and we will know the price of each item.
We will use this information to fill missing prices based on the item name.

First, we will create a mapping from item names to their price using the top items DataFrame (z).
Then, we will fill the missing prices in the original DataFrame by looking up each item in this mapping.

This will ensure all missing prices are replaced with the correct values for each item.

In [ ]:
map_price = dict(zip(z['Item'].str.lower(), z['Price Per Unit']))

In [ ]:
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].str.lower().map(map_price))

# Decrease in Missing price Count

By filling the missing values, we will reduce the number of null price from 533 to 23.

In [ ]:
df['Price Per Unit'].isna().sum()

np.int64(54)

# Unfilled Items
We will observe that 333 items are missing.

In [ ]:
df[df['Item'].isna()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
30,TXN_1736287,NaN,5.0,2.0,10.0,Digital Wallet,NaN,2023-06-02
61,TXN_8051289,NaN,1.0,3.0,3.0,NaN,In-store,2023-10-09
72,TXN_6044979,NaN,1.0,1.0,1.0,Cash,In-store,2023-12-08
89,TXN_4132730,NaN,5.0,1.0,5.0,NaN,In-store,2023-03-12
...,...,...,...,...,...,...,...,...
9820,TXN_8751702,NaN,5.0,NaN,15.0,Cash,NaN,2023-02-13
9855,TXN_3740505,NaN,2.0,1.5,3.0,NaN,NaN,2023-11-21
9876,TXN_3105633,NaN,1.0,2.0,2.0,NaN,In-store,2023-03-30
9885,TXN_4659954,NaN,3.0,4.0,12.0,Credit Card,In-store,NaT


# Standardize & Complete the 'Item' Field

Perform these operations on the 'Item' column:

1. Convert the column to string type, remove whitespace from both ends of each value, and transform all text to uppercase.
2. Replace any clearly invalid or placeholder entries (e.g. 'UNKNOWN', 'ERROR', 'NAN', blank/empty strings) with NaN.
3. Build a lookup table that associates each distinct 'Price Per Unit' with the corresponding 'Item' name, based on the clean, high-quality data found in the top items DataFrame (z).
4. Fill any missing 'Item' values by using the price in that row to look up the most appropriate item name from the created mapping.

In [ ]:
df['Item'] = df['Item'].astype(str).str.strip().str.lower().replace(['unknown', 'error', 'nan', ''], np.nan)

In [ ]:
price_to_item = dict(zip(z['Price Per Unit'], z['Item']))


df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(price_to_item))

# Unfilled Items
We will observe that 54 items are missing

In [ ]:
df['Item'].isna().sum()

np.int64(54)

In [ ]:
df.isna().sum()

,0
Transaction ID,0
Item,54
Quantity,479
Price Per Unit,54
Total Spent,502
Payment Method,2579
Location,3265
Transaction Date,460


## Unfilled Total Spent

We will observe that 502 items are missing.

# Impute 'Total Spent'
The 'Total Spent' column has some null values.
We will calculate and fill them using the formula: Total Spent = Price * Quantity.

In [ ]:
df['Total Spent'] = df['Total Spent'].fillna(0)

In [ ]:
df[df['Total Spent'] == 0]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2,TXN_4271903,cookie,4.0,1.0,0.0,Credit Card,In-store,2023-07-19
25,TXN_7958992,smoothie,3.0,4.0,0.0,UNKNOWN,UNKNOWN,2023-12-13
31,TXN_8927252,Cookie,2.0,1.0,0.0,Credit Card,ERROR,2023-11-06
42,TXN_6650263,tea,2.0,1.5,0.0,NaN,Takeaway,2023-01-10
65,TXN_4987129,sandwich,3.0,4.0,0.0,NaN,In-store,2023-10-20
...,...,...,...,...,...,...,...,...
9893,TXN_3809533,juice,2.0,3.0,0.0,Digital Wallet,Takeaway,2023-02-02
9954,TXN_1191659,coffee,4.0,2.0,0.0,Credit Card,In-store,2023-11-21
9977,TXN_5548914,juice,2.0,3.0,0.0,Digital Wallet,In-store,2023-11-04
9988,TXN_9594133,cake,5.0,3.0,0.0,ERROR,NaN,NaT


In [ ]:
df['Total Spent'] = np.where(df['Total Spent'] == 0 , df['Price Per Unit'] * df['Quantity'] , df['Total Spent'])

The number of null items got reduce from 502 to 23.

In [ ]:
df['Total Spent'].isna().sum()

np.int64(23)

In [ ]:
df.isna().sum()

,0
Transaction ID,0
Item,54
Quantity,479
Price Per Unit,54
Total Spent,23
Payment Method,2579
Location,3265
Transaction Date,460


# Fill Missing 'Quantity' Values
We will fill missing values in the 'Quantity' column using the formula:
Quantity = Total Spent ÷ Price Per Unit.first we will fill missing values with 0

In [ ]:
df['Quantity'] = df['Quantity'].fillna(0)

In [ ]:
df[df['Quantity'] == 0]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
20,TXN_3522028,smoothie,0.0,4.0,20.0,Cash,In-store,2023-04-04
55,TXN_5522862,cookie,0.0,1.0,2.0,Credit Card,Takeaway,2023-03-19
57,TXN_2080895,cake,0.0,3.0,3.0,Digital Wallet,In-store,2023-04-19
66,TXN_8501819,juice,0.0,3.0,6.0,Cash,NaN,2023-03-30
117,TXN_2148617,juice,0.0,3.0,9.0,Digital Wallet,UNKNOWN,2023-01-10
...,...,...,...,...,...,...,...,...
9932,TXN_8502079,tea,0.0,1.5,3.0,Cash,NaN,2023-04-20
9935,TXN_9778251,tea,0.0,1.5,6.0,NaN,Takeaway,2023-11-09
9944,TXN_7495283,cake,0.0,3.0,15.0,Credit Card,Takeaway,2023-04-14
9957,TXN_6487003,coffee,0.0,2.0,8.0,Credit Card,Takeaway,2023-11-15


In [ ]:
df['Quantity'] = np.where(df['Quantity'] == 0 , df['Total Spent'] / df['Price Per Unit'] , df['Quantity'])

In [ ]:
df['Quantity'].isna().sum()

np.int64(23)

# REPEATING THE SAME STEPS TO FIND REMAINING VALUES



# Re-run Cells
We will re-run some cells used earlier because we have filled values that other calculations depend on.
By re-running these cells, we will fill the remaining missing values correctly.

In [ ]:
df['Price Per Unit']=df['Price Per Unit'].fillna(0)

df[df['Price Per Unit']==0].head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
118,TXN_4633784,NaN,5.0,0.0,15.0,NaN,In-store,2023-02-06
151,TXN_4031509,NaN,4.0,0.0,16.0,Credit Card,Takeaway,2023-01-04
289,TXN_3495950,NaN,4.0,0.0,6.0,Credit Card,In-store,2023-02-19
334,TXN_2523298,NaN,4.0,0.0,6.0,ERROR,In-store,2023-03-25
550,TXN_4186681,NaN,4.0,0.0,6.0,Digital Wallet,NaN,2023-05-24


In [ ]:
df['Price Per Unit']=np.where(df['Price Per Unit']==0,df['Total Spent']/df['Quantity'],df['Price Per Unit'])

In [ ]:
df[df['Price Per Unit']==0]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date


Price got reduce to 6

In [ ]:
df['Price Per Unit'].isna().sum()

np.int64(6)

We will re-run some cells used earlier because we have filled values that other calculations depend on. By re-running these cells, we will fill the remaining missing values correctly.

In [ ]:
df['Item'] = df['Item'].astype(str).str.strip().str.lower().replace(['unknown', 'error', 'nan', ''], np.nan)

In [ ]:
price_to_item = dict(zip(z['Price Per Unit'], z['Item']))


df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(price_to_item))

Item got reduced to 6

In [ ]:
df['Item'].isna().sum()

np.int64(6)

In [ ]:
df.isna().sum()

,0
Transaction ID,0
Item,6
Quantity,23
Price Per Unit,6
Total Spent,23
Payment Method,2579
Location,3265
Transaction Date,460


Some values in the 'Quantity' column could not be filled because both dependent
columns, 'Total Spent' and 'Price Per Unit', are NaN.